# Real vs Fake Faces

We are going to `Fake-Vs-Real-Faces`. The Dataset contains two types of image which are: `Fake faces` image collected from the website [thispersondoesnotexist.com](thispersondoesnotexist.com). , `Real faces` images are re collected through the API of the website [Unsplash](https://unsplash.com/) and then the faces are cropped out of using OpenCV library.

Here some additional information about the dataset:
- Total number of images: 1288
- Number of "Fake" faces: 700
- Number of "Real" faces: 589

### 1. Import Libraries 

In [1]:
import torch 
import torch.nn as nn
from torch.nn import Linear, Dropout, ReLU
from torchvision import transforms
from torch.utils.data import DataLoader, Subset, random_split, Dataset
from torchvision.datasets import ImageFolder # Stream data from images stored in folders
import torch.optim as optim
import torch.nn.functional as F
from torchmetrics.classification import BinaryPrecision, BinaryRecall, BinaryF1Score
from torchinfo import summary

from sklearn.model_selection import StratifiedKFold, train_test_split
import os # allow s to access files
import numpy as np
from PIL import Image # Allows us to Load Images
from collections import Counter # Utility function to give us the counts of unique items in an iterable
import matplotlib.pyplot as plt
import shutil

## 2. Reconstruct the Folder structure of the Data

In [2]:
torch.manual_seed(42)

Before loading, let's reconstruct the data folder structure to give us comfort when using the `ImageFolder` class

In [3]:
def copy_image(des_dir, split_name, cls, file_list, cls_path):

    """
    des_dir: data_split
    split_name: can be "train" or "test"
    cls and cls_path: is class which can be "fake" or "real"
    file_list: list of images files

    save a copy of images from folder 'cls_path/img' into folder called 'des_dir/split_name/cls/img' 
    """
    
    for img in file_list:
        src = os.path.join(cls_path, img)
        dest = os.path.join(des_dir, split_name, cls, img)
        shutil.copy2(src, dest)

In [4]:

def split_and_save_to_disk(src_dir, des_dir):

    """
    reconstruct the folder structure 
    from:
        data/
            fake/
            real

    to:
        data_split/
            train/
                fake/
                real/
            test/
                fake/
                real/
    
    """
    
    # # splite ratios
    train_ratio = 0.8
    test_ratio = 0.2

    if os.path.exists(des_dir):
        print(f"{src_dir} already exists")
    else:
        print("reconstruct the data folder structure")

        # create folder 
        for split in ['train', 'test']:
            for cls in os.listdir(src_dir):
                os.makedirs(os.path.join(des_dir, split, cls), exist_ok=True)  # make it similar to this (des_dir/train/fake) and  (des_dir/train/real)


        # Process each class folder 
        for cls in os.listdir(src_dir):
            cls_path = os.path.join(src_dir, cls)
            if not os.path.isdir(cls_path):
                continue


            images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

            # split train and (test and val)
            train_data, test_data = train_test_split(images, test_size=test_ratio, random_state=42)


            # splitting the data 
            print(f"splitting class '{cls}':\n train:{len(train_data)}\n test:{len(test_data)}")

            # des_dir, split_name, cls, file_list, cls_path
            copy_image(des_dir, "train", cls, train_data, cls_path)
            copy_image(des_dir, "test", cls, test_data, cls_path)
            
        print("Copying proces is finished")
        

## 3. Load and Prepocess the data

First Let's define the transforms pipeline for `data augmentatin` and `preprocessing` steps

In [5]:
split_and_save_to_disk("data", "data_split")

data already exists


In [6]:
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]

# for data preprocessing and data augmentation for trianing data
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])


test_valid_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),

])

Let's load our images using `ImageFolder`. This class inherits from `DatasetFolder`  class. So, we can override the classes to create custom datasets as well. Because we need to custome dataset class becasue `ImageFolder` apply a single tansform on the whole dataset while we need to apply custome `transform` on each subset in the dataset.

In [7]:
path_to_train = './split_data/train'
path_to_test = './split_data/test'

train_data = ImageFolder(root=path_to_train, transform=None)
test_data = ImageFolder(root=path_to_test, transform=test_valid_transforms)

class CustomSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    
    def __getitem__(self, idx):
        
        actual_idx = self.subset.indices[idx]
        
        image_path, label = self.subset.dataset.samples[actual_idx]
        image = self.subset.dataset.loader(image_path) 
        
        
        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)
        
        return image, label
    
    def __len__(self):
        return len(self.subset)
        
# Create the DataLoders
BATCH_SIZE = 16
# train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=True)

## 3. Build Helper functions

Before starting building the model we will build a helper function to save our models

In [8]:
def save_model(epochs, model, optimizer, loss_fun, model_name):

    """
    Function to save the trained model to disk
    """
    torch.save({
        'epoch': epochs,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss_fun,
    }, f'outputs/{model_name}.pth')

The next function is for saving the loss and accuracy plots after the training completes

In [9]:
def save_plots(train_acc, valid_acc, test_acc, train_loss, valid_loss, test_loss):

    """
    Function to save the loss and accurcy plots to disk
    """

    # accuracy plots
    plt.figure(figsize=(10, 7))
    plt.plot(train_acc, color='green', linestyle='-', label='train accuracy')
    plt.plot(valid_acc, color='red', linestyle='-', label='validation accuracy')
    plt.plot(test_acc, color='cyan', linestyle='-', label='test accuracy')
    plt.xlabel('EPOCHs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig('outputs/accuracy.png')

    # loss plots
    plt.figure(figsize=(10, 7))
    plt.plot(train_loss, color='orange', linestyle='-', label='train loss')
    plt.plot(valid_loss, color='blue', linestyle='-', label='valid loss')
    plt.plot(test_loss, color='magenta', linestyle='-', label='test loss')
    plt.xlabel('EPOCHs')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig('outputs/loss.png')

## 4. Prepparing the Data Loaders and Stratified KFold

Now we will build our `data loader` and `stratified KFold` spliter for that we need for training.

In [10]:
N_SPLITs = 5
SEED = 42

In [108]:
def train(model, training_data, X, y, optimizer, loss_fn, loss_aux, Epochs = 5, N_SPLITs = 4):

    """
    Train the given model
    """


    
    skf = StratifiedKFold(n_splits=N_SPLITs, shuffle=True, random_state=SEED)
    stratified_scores = []

    # folds results 
    all_fold_results = []

    # Lists to collect metrics
    train_acc_list, valid_acc_list = [], []
    train_loss_list, valid_loss_list = [], []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):



        # Initialize fold-specific metrics
        fold_train_loss = []
        fold_val_loss = []
        fold_train_acc = []
        fold_val_acc = []
        
        # Create the Data Loaders
        train_subset = Subset(training_data, train_idx)
        val_subset = Subset(training_data, valid_idx)


        # Apply transforms dynamically
        train_data_fold = CustomSubset(train_subset, transform=train_transforms)
        val_data_fold = CustomSubset(val_subset, transform=test_valid_transforms)
    
        # Dataloaders
        train_loader = DataLoader(train_data_fold, batch_size=BATCH_SIZE, shuffle=True)
        valid_loader = DataLoader(val_data_fold, batch_size=BATCH_SIZE, shuffle=False)

        ### Training ###
        model.train() 
        print("Training")
        for epoch in range(Epochs):

            train_loss, correct, total = 0.0, 0, 0
            
            for images, labels in train_loader:
                labels = labels.unsqueeze(1).float()

                outputs = model(images)
                if isinstance(outputs, tuple):
                    main_out, aux2_out, aux1_out = outputs
                    loss_main = loss_fn(main_out, labels)
                    loss_aux1 = loss_fn(aux1_out, labels)
                    loss_aux2 = loss_fn(aux2_out, labels)
                    loss = loss_main + 0.3 * (loss_aux1 + loss_aux2)
                    preds = (torch.sigmoid(main_out) > 0.5).float()
                else:
                    loss = loss_fn(outputs, labels)
                    preds = (torch.sigmoid(outputs) > 0.5).float()
       
                loss.backward() # backpagation

                train_loss += loss.item()
                
                optimizer.step() # update the weight
                optimizer.zero_grad() # set the gradient to zero
                
                correct += (preds == labels).sum().item()
                total += labels.size(0)


            # calculate the accuracy and loss for the current epoch
            train_accuracy = correct / total
            epoch_train_loss = train_loss / len(train_loader)


            ### Validation ###
            model.eval()
            print("Evaluting")
            val_loss, correct, total = 0.0, 0, 0
            with torch.no_grad():
                for images, labels in valid_loader:
                    predictions = model(images)
                    
                    if isinstance(predictions, tuple):
                        predictions = predictions[0]
                    else:
                        predictions = predictions
                        
                    labels = labels.unsqueeze(1).float()
                    val_loss += loss_fn(predictions, labels).item()
                    
                    preds = (torch.sigmoid(predictions) > 0.5).float()
                    correct += (preds == labels).sum().item()
                    total += labels.size(0)

                    
            val_accuracy = correct / total
            epoch_val_loss = val_loss / len(valid_loader)
     

        
            fold_train_loss.append(train_loss)
            fold_val_loss.append(val_loss)
            fold_train_acc.append(train_accuracy)
            fold_val_acc.append(val_accuracy)
            
            print(f"Epoch {epoch+1}/{Epochs} - Train Loss: {epoch_train_loss:.4f}, Train Acc: {train_accuracy:.4f}, Val Loss: {epoch_val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

            
        # Store metrics for this fold
        all_fold_results.append({
            'fold': fold + 1,
            'train_loss': fold_train_loss,
            'val_loss': fold_val_loss,
            'train_acc': fold_train_acc,
            'val_acc': fold_val_acc
        })


    avg_val_acc = sum(r['val_acc'][-1] for r in all_fold_results) / N_SPLITs
    avg_val_loss = sum(r['val_loss'][-1] for r in all_fold_results) / N_SPLITs
    print(f"\nAverage Validation Accuracy across folds: {avg_val_acc:.4f}")
    print(f"Average Validation Loss across folds: {avg_val_loss:.4f}")

    return all_fold_results
        
              
    

## 4, 5. Build and Train the Models

> ### 4.1 GoogleNet

In [109]:
# Load the googlenet model and weights with pre-trian ImageNet weights
from torchvision.models import googlenet
from torchvision.models import GoogLeNet_Weights

# initialize the pretrained googlenet
googlenet_forzen = googlenet(weights=GoogLeNet_Weights.DEFAULT, aux_logits=True)

googlenet_forzen.eval()

D:\Semi1\Select Topic In AI\Real vs Fake faces\.venv\lib\site-packages\torchvision\models\googlenet.py:341: UserWarning: auxiliary heads in the pretrained googlenet model are NOT pretrained, so make sure to train them
  warnings.warn(


GoogLeNet(
  (conv1): BasicConv2d(
    (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (conv2): BasicConv2d(
    (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (conv3): BasicConv2d(
    (conv): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(192, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (inception3a): Inception(
    (branch1): BasicConv2d(
      (conv): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track

> ### 4.2 Freeze early layers 

In [110]:
for parameter in googlenet_forzen.parameters():
    parameter.requires_grad = False


> ### 4.3 Modfiy the googlenet (output layer)

In [111]:
num_classes = 1

In [112]:
googlenet_forzen.fc = nn.Linear(googlenet_forzen.fc.in_features, num_classes)
googlenet_forzen.aux1.fc2 = nn.Linear(1024, num_classes)
googlenet_forzen.aux2.fc2 = nn.Linear(1024, num_classes)

> ### 4.4 Get model summary

In [113]:
summary(googlenet_forzen,
        input_size=(16, 3, 224, 224),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=18,
        row_settings=["var_names"]
)

Layer (type (var_name))                  Input Shape        Output Shape       Param #            Trainable
GoogLeNet (GoogLeNet)                    [16, 3, 224, 224]  [16, 1]            4,332,034          Partial
├─BasicConv2d (conv1)                    [16, 3, 224, 224]  [16, 64, 112, 112] --                 False
│    └─Conv2d (conv)                     [16, 3, 224, 224]  [16, 64, 112, 112] (9,408)            False
│    └─BatchNorm2d (bn)                  [16, 64, 112, 112] [16, 64, 112, 112] (128)              False
├─MaxPool2d (maxpool1)                   [16, 64, 112, 112] [16, 64, 56, 56]   --                 --
├─BasicConv2d (conv2)                    [16, 64, 56, 56]   [16, 64, 56, 56]   --                 False
│    └─Conv2d (conv)                     [16, 64, 56, 56]   [16, 64, 56, 56]   (4,096)            False
│    └─BatchNorm2d (bn)                  [16, 64, 56, 56]   [16, 64, 56, 56]   (128)              False
├─BasicConv2d (conv3)                    [16, 64, 56, 56]   [

> ### 4.4 Train the Model

In [114]:
# Specifiy learing rate and number of Epochs
googlenet_LR = 0.001
googlenet_Epochs = 5
googlenet_splits = 5
# Define loss and optimizer
googlenet_loss_fun = nn.BCEWithLogitsLoss()
googlenet_optimizer = torch.optim.Adam(googlenet_forzen.parameters(), lr=googlenet_LR)


train_X = [train_data.samples[img][0] for img in range(len(train_data.samples))]
train_y = [train_data.samples[label][1] for label in range(len(train_data.samples))]

skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
BATCH_SIZE = 16

In [115]:
result = train(googlenet_forzen, train_data, train_X, train_y, googlenet_optimizer, googlenet_loss_fun, googlenet_Epochs, N_SPLITs = googlenet_splits)

Training
Evaluting
Epoch 1/5 - Train Loss: 0.8663, Train Acc: 0.7791, Val Loss: 0.3745, Val Acc: 0.8986
Evaluting
Epoch 2/5 - Train Loss: 0.3032, Train Acc: 0.9660, Val Loss: 0.2273, Val Acc: 0.9710
Evaluting
Epoch 3/5 - Train Loss: 0.2155, Train Acc: 0.9648, Val Loss: 0.1625, Val Acc: 0.9807
Evaluting
Epoch 4/5 - Train Loss: 0.1603, Train Acc: 0.9842, Val Loss: 0.1592, Val Acc: 0.9614
Evaluting
Epoch 5/5 - Train Loss: 0.1364, Train Acc: 0.9806, Val Loss: 0.1280, Val Acc: 0.9710
Training
Evaluting
Epoch 1/5 - Train Loss: 0.3734, Train Acc: 0.9576, Val Loss: 0.1230, Val Acc: 0.9854
Evaluting
Epoch 2/5 - Train Loss: 0.0952, Train Acc: 0.9891, Val Loss: 0.1030, Val Acc: 0.9903
Evaluting
Epoch 3/5 - Train Loss: 0.0840, Train Acc: 0.9891, Val Loss: 0.1007, Val Acc: 0.9951
Evaluting
Epoch 4/5 - Train Loss: 0.0845, Train Acc: 0.9830, Val Loss: 0.0877, Val Acc: 0.9903
Evaluting
Epoch 5/5 - Train Loss: 0.0725, Train Acc: 0.9903, Val Loss: 0.0850, Val Acc: 0.9903
Training
Evaluting
Epoch 1/5 - T

> ### 5.1 Full Fine-Tuning

In [116]:
# Load the googlenet model and weights with pre-trian ImageNet weights
from torchvision.models import googlenet
from torchvision.models import GoogLeNet_Weights

# initialize the pretrained googlenet
googlenet_full = googlenet(weights=GoogLeNet_Weights.DEFAULT, aux_logits=True)

googlenet_full.eval()

D:\Semi1\Select Topic In AI\Real vs Fake faces\.venv\lib\site-packages\torchvision\models\googlenet.py:341: UserWarning: auxiliary heads in the pretrained googlenet model are NOT pretrained, so make sure to train them
  warnings.warn(


GoogLeNet(
  (conv1): BasicConv2d(
    (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (conv2): BasicConv2d(
    (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (conv3): BasicConv2d(
    (conv): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(192, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (inception3a): Inception(
    (branch1): BasicConv2d(
      (conv): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track

> ### Freeze 

In [121]:
googlenet_full.fc = nn.Linear(googlenet_full.fc.in_features, num_classes)
googlenet_full.aux1.fc2 = nn.Linear(googlenet_full.aux1.fc2.in_features, num_classes)
googlenet_full.aux2.fc2 = nn.Linear(googlenet_full.aux2.fc2.in_features, num_classes)

> ### 5.2 Get model summary

In [122]:
summary(googlenet_full,
        input_size=(16, 3, 224, 224),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=18,
        row_settings=["var_names"]
)

Layer (type (var_name))                  Input Shape        Output Shape       Param #            Trainable
GoogLeNet (GoogLeNet)                    [16, 3, 224, 224]  [16, 1]            4,332,034          True
├─BasicConv2d (conv1)                    [16, 3, 224, 224]  [16, 64, 112, 112] --                 True
│    └─Conv2d (conv)                     [16, 3, 224, 224]  [16, 64, 112, 112] 9,408              True
│    └─BatchNorm2d (bn)                  [16, 64, 112, 112] [16, 64, 112, 112] 128                True
├─MaxPool2d (maxpool1)                   [16, 64, 112, 112] [16, 64, 56, 56]   --                 --
├─BasicConv2d (conv2)                    [16, 64, 56, 56]   [16, 64, 56, 56]   --                 True
│    └─Conv2d (conv)                     [16, 64, 56, 56]   [16, 64, 56, 56]   4,096              True
│    └─BatchNorm2d (bn)                  [16, 64, 56, 56]   [16, 64, 56, 56]   128                True
├─BasicConv2d (conv3)                    [16, 64, 56, 56]   [16, 192, 

> ### 5.3 Train the model

In [124]:
googlenet_full_optimizer = torch.optim.Adam(googlenet_full.parameters(), lr=googlenet_LR)

In [125]:
result_full = train(googlenet_full, train_data, train_X, train_y, googlenet_full_optimizer, googlenet_loss_fun, googlenet_Epochs, N_SPLITs = googlenet_splits)

Training
Evaluting
Epoch 1/5 - Train Loss: 0.1387, Train Acc: 0.9612, Val Loss: 0.0026, Val Acc: 1.0000
Evaluting
Epoch 2/5 - Train Loss: 0.6494, Train Acc: 0.6396, Val Loss: 0.6904, Val Acc: 0.5411
Evaluting
Epoch 3/5 - Train Loss: 0.6911, Train Acc: 0.5340, Val Loss: 0.6901, Val Acc: 0.5411
Evaluting
Epoch 4/5 - Train Loss: 0.6908, Train Acc: 0.5437, Val Loss: 0.6901, Val Acc: 0.5411
Evaluting
Epoch 5/5 - Train Loss: 0.6912, Train Acc: 0.5437, Val Loss: 0.6906, Val Acc: 0.5411
Training
Evaluting
Epoch 1/5 - Train Loss: 0.0323, Train Acc: 0.9976, Val Loss: 0.0007, Val Acc: 1.0000
Evaluting
Epoch 2/5 - Train Loss: 0.4262, Train Acc: 0.7794, Val Loss: 0.6953, Val Acc: 0.4563
Evaluting
Epoch 3/5 - Train Loss: 0.6938, Train Acc: 0.5394, Val Loss: 0.7430, Val Acc: 0.5437
Evaluting
Epoch 4/5 - Train Loss: 0.6943, Train Acc: 0.5430, Val Loss: 0.6918, Val Acc: 0.5437
Evaluting
Epoch 5/5 - Train Loss: 0.6917, Train Acc: 0.5345, Val Loss: 0.6942, Val Acc: 0.4563
Training
Evaluting
Epoch 1/5 - T

> ### 6.1 Partial Fine-Tuning

In [130]:
# Load the googlenet model and weights with pre-trian ImageNet weights
from torchvision.models import googlenet
from torchvision.models import GoogLeNet_Weights

# initialize the pretrained googlenet
googlenet_partial = googlenet(weights=GoogLeNet_Weights.DEFAULT, aux_logits=True)

googlenet_partial.eval()

D:\Semi1\Select Topic In AI\Real vs Fake faces\.venv\lib\site-packages\torchvision\models\googlenet.py:341: UserWarning: auxiliary heads in the pretrained googlenet model are NOT pretrained, so make sure to train them
  warnings.warn(


GoogLeNet(
  (conv1): BasicConv2d(
    (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (conv2): BasicConv2d(
    (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (conv3): BasicConv2d(
    (conv): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(192, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (inception3a): Inception(
    (branch1): BasicConv2d(
      (conv): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track

> ### 6.2 Modfiy the GoogleNet later layers 

In [140]:
for parameter in googlenet_partial.parameters():
    parameter.requires_grad = False

In [141]:
# Get all parameter names and unfreeze last few layers
all_params = list(googlenet_partial.named_parameters())
for name, param in all_params[-8:]:
    param.requires_grad = True
    print(f"Unfrozen: {name}")

Unfrozen: aux2.conv.bn.weight
Unfrozen: aux2.conv.bn.bias
Unfrozen: aux2.fc1.weight
Unfrozen: aux2.fc1.bias
Unfrozen: aux2.fc2.weight
Unfrozen: aux2.fc2.bias
Unfrozen: fc.weight
Unfrozen: fc.bias


In [142]:
googlenet_partial.fc = nn.Linear(googlenet_partial.fc.in_features, num_classes)
googlenet_partial.aux1.fc2 = nn.Linear(googlenet_partial.aux1.fc2.in_features, num_classes)
googlenet_partial.aux2.fc2 = nn.Linear(googlenet_partial.aux2.fc2.in_features, num_classes)

> ### 6.3 Get model summary

In [143]:
summary(googlenet_partial,
        input_size=(16, 3, 224, 224),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=18,
        row_settings=["var_names"]
)

Layer (type (var_name))                  Input Shape        Output Shape       Param #            Trainable
GoogLeNet (GoogLeNet)                    [16, 3, 224, 224]  [16, 1]            4,332,034          Partial
├─BasicConv2d (conv1)                    [16, 3, 224, 224]  [16, 64, 112, 112] --                 False
│    └─Conv2d (conv)                     [16, 3, 224, 224]  [16, 64, 112, 112] (9,408)            False
│    └─BatchNorm2d (bn)                  [16, 64, 112, 112] [16, 64, 112, 112] (128)              False
├─MaxPool2d (maxpool1)                   [16, 64, 112, 112] [16, 64, 56, 56]   --                 --
├─BasicConv2d (conv2)                    [16, 64, 56, 56]   [16, 64, 56, 56]   --                 False
│    └─Conv2d (conv)                     [16, 64, 56, 56]   [16, 64, 56, 56]   (4,096)            False
│    └─BatchNorm2d (bn)                  [16, 64, 56, 56]   [16, 64, 56, 56]   (128)              False
├─BasicConv2d (conv3)                    [16, 64, 56, 56]   [

> ### 6.4 Train the model

In [144]:
googlenet_partial_optimizer = torch.optim.Adam(googlenet_partial.parameters(), lr=googlenet_LR)

In [145]:
result_partial = train(googlenet_partial, train_data, train_X, train_y, googlenet_partial_optimizer, googlenet_loss_fun, googlenet_Epochs, N_SPLITs = googlenet_splits)

Training
Evaluting
Epoch 1/5 - Train Loss: 0.7890, Train Acc: 0.8022, Val Loss: 0.3560, Val Acc: 0.9469
Evaluting
Epoch 2/5 - Train Loss: 0.3099, Train Acc: 0.9563, Val Loss: 0.2537, Val Acc: 0.9372
Evaluting
Epoch 3/5 - Train Loss: 0.2139, Train Acc: 0.9697, Val Loss: 0.2018, Val Acc: 0.9372
Evaluting
Epoch 4/5 - Train Loss: 0.1613, Train Acc: 0.9830, Val Loss: 0.1567, Val Acc: 0.9614
Evaluting
Epoch 5/5 - Train Loss: 0.1411, Train Acc: 0.9769, Val Loss: 0.1449, Val Acc: 0.9614
Training
Evaluting
Epoch 1/5 - Train Loss: 0.2753, Train Acc: 0.9612, Val Loss: 0.1270, Val Acc: 0.9757
Evaluting
Epoch 2/5 - Train Loss: 0.1041, Train Acc: 0.9745, Val Loss: 0.1072, Val Acc: 0.9903
Evaluting
Epoch 3/5 - Train Loss: 0.0856, Train Acc: 0.9903, Val Loss: 0.0960, Val Acc: 0.9806
Evaluting
Epoch 4/5 - Train Loss: 0.0783, Train Acc: 0.9915, Val Loss: 0.0936, Val Acc: 0.9903
Evaluting
Epoch 5/5 - Train Loss: 0.0747, Train Acc: 0.9867, Val Loss: 0.0840, Val Acc: 0.9903
Training
Evaluting
Epoch 1/5 - T

## 7. Test the models

In [146]:
from torchmetrics.classification import BinaryPrecision, BinaryRecall, BinaryF1Score

def test(model, test_loader, loss_fn):
    """
    Test the model and return testing metrics
    """
    model.eval()
    print("Testing")
    
    val_loss, correct, total = 0.0, 0, 0
    y_pred, y_true = [], []
    
    # Initialize metrics
    precision = BinaryPrecision()
    recall = BinaryRecall()
    f1 = BinaryF1Score()
    
    with torch.no_grad():
        for images, labels in test_loader:
            predictions = model(images)
            labels = labels.unsqueeze(1).float()
            
   
            val_loss += loss_fn(predictions, labels).item()
            

            preds = (torch.sigmoid(predictions) > 0.5).float()
            
            y_pred.append(preds)
            y_true.append(labels)
            
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    y_pred = torch.cat(y_pred, dim=0)
    y_true = torch.cat(y_true, dim=0)
    
    # Metrics
    test_accuracy = correct / total
    test_loss = val_loss / len(test_loader)
    test_precision = precision(y_pred, y_true).item()
    test_recall = recall(y_pred, y_true).item()
    test_f1 = f1(y_pred, y_true).item()
    
    return {
        'test_loss': test_loss,
        'test_accuracy': test_accuracy,
        'test_precision': test_precision,
        'test_recall': test_recall,
        'test_f1': test_f1,
    }

In [150]:
googlenet_forzen_test_results = test(googlenet_forzen, test_loader, googlenet_loss_fun)

Testing


In [151]:
googlenet_full_test_results = test(googlenet_full, test_loader, googlenet_loss_fun)

Testing


In [152]:
googlenet_partial_test_results = test(googlenet_partial, test_loader, googlenet_loss_fun)

Testing


In [154]:
googlenet_forzen_test_results

{'test_loss': 0.17332936797498,
 'test_accuracy': 0.9457364341085271,
 'test_precision': 1.0,
 'test_recall': 0.8813559412956238,
 'test_f1': 0.9369369149208069}

In [155]:
googlenet_partial_test_results

{'test_loss': 0.08422821139295895,
 'test_accuracy': 0.9457364341085271,
 'test_precision': 1.0,
 'test_recall': 0.8813559412956238,
 'test_f1': 0.9369369149208069}

In [153]:
googlenet_full_test_results

{'test_loss': 0.14014589719267356,
 'test_accuracy': 0.937984496124031,
 'test_precision': 0.8805969953536987,
 'test_recall': 1.0,
 'test_f1': 0.9365079402923584}

In [158]:
avg_val_acc = sum(r['val_acc'][-1] for r in result) / googlenet_splits
avg_val_loss = sum(r['val_loss'][-1] for r in result) / googlenet_splits
avg_train_acc = sum(r['train_acc'][-1] for r in result) / googlenet_splits
avg_train_loss = sum(r['train_loss'][-1] for r in result) / googlenet_splits
print(f"googlenet average Validation Accuracy across folds: {avg_val_acc:.4f}")
print(f"googlenet average Validation Loss across folds: {avg_val_loss:.4f}")
print(f"\ngooglenet average Training Accuracy across folds: {avg_train_acc:.4f}")
print(f"googlenet average Traing Loss across folds: {avg_train_loss:.4f}")

googlenet average Validation Accuracy across folds: 0.9845
googlenet average Validation Loss across folds: 1.0169

googlenet average Training Accuracy across folds: 0.9871
googlenet average Traing Loss across folds: 3.7728


In [159]:
avg_val_acc = sum(r['val_acc'][-1] for r in result_full) / googlenet_splits
avg_val_loss = sum(r['val_loss'][-1] for r in result_full) / googlenet_splits
avg_train_acc = sum(r['train_acc'][-1] for r in result_full) / googlenet_splits
avg_train_loss = sum(r['train_loss'][-1] for r in result_full) / googlenet_splits
print(f"googlenet full fin-tuning average Validation Accuracy across folds: {avg_val_acc:.4f}")
print(f"googlenet full fine-tuning average Validation Loss across folds: {avg_val_loss:.4f}")
print(f"\nVGG16 full fine-tuning average Training Accuracy across folds: {avg_train_acc:.4f}")
print(f"VGG16 full fine-tuning average Traing Loss across folds: {avg_train_loss:.4f}")

googlenet full-fine tuning average Validation Accuracy across folds: 0.5558
googlenet full-fine tuning average Validation Loss across folds: 9.5173

VGG16 full-fine tuning average Training Accuracy across folds: 0.6287
VGG16 full-fine tuning average Traing Loss across folds: 30.2374


In [161]:
avg_val_acc = sum(r['val_acc'][-1] for r in result_partial) / googlenet_splits
avg_val_loss = sum(r['val_loss'][-1] for r in result_partial) / googlenet_splits
avg_train_acc = sum(r['train_acc'][-1] for r in result_partial) / googlenet_splits
avg_train_loss = sum(r['train_loss'][-1] for r in result_partial) / googlenet_splits
print(f"googlenet partial fin-tuning average Validation Accuracy across folds: {avg_val_acc:.4f}")
print(f"googlenet partial fin-tuning average Validation Loss across folds: {avg_val_loss:.4f}")
print(f"\nVGG16 partial fin-tuning average Training Accuracy across folds: {avg_train_acc:.4f}")
print(f"VGG16 partial fin-tuning average Traing Loss across folds: {avg_train_loss:.4f}")

googlenet partial fin-tuning average Validation Accuracy across folds: 0.9816
googlenet partial fin-tuning average Validation Loss across folds: 1.0353

VGG16 partial fin-tuning average Training Accuracy across folds: 0.9869
VGG16 partial fin-tuning average Traing Loss across folds: 3.7514
